# dev/camera_bottom — present a disc to the bottom camera

Sandbox agenda: spawn a disc into `in_1[A1]`, suction-pick it, and
drive to the station camera's present pose — `inspector` over
`inspection_horizontal_1` — then hold there so the view/lighting can
be checked. Present REQUIRES a held item (place_setting raises "no
item in the gripper" otherwise), so the pick is not optional.

Pick, present and return are separate cells.

**Scene, layout, calibration and recipes are the MAIN program's** — the
pair from `launch.yaml` plus `recipes.j2`. Start jupyter in this
folder; the init cell walks up to the project root on its own.


# Start

In [1]:
# ── START the notebook 3D-viewer server on port 8000 (idempotent) ──
# Port 5000 belongs to the orchestrator (gui/server.py) — that one does
# NOT stream the viewer; workspace/server.py does.
import subprocess, urllib.request, time, os
from pathlib import Path

def viewer_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/', timeout=2)
        return True
    except Exception:
        return False

if not viewer_up():
    srv = Path.home() / 'Downloads/workspace/workspace/server.py'
    try:
        log = open(f'/tmp/workspace_viewer.{os.getuid()}.log', 'ab')
    except OSError:
        log = subprocess.DEVNULL
    subprocess.Popen(['sudo', 'env', 'PORT=8000', 'python3', str(srv)],
                     cwd=srv.parent, stdout=log, stderr=log,
                     start_new_session=True)   # survives kernel restarts
    while not viewer_up():
        time.sleep(0.5)
import socket as _sk
_s=_sk.socket(_sk.AF_INET,_sk.SOCK_DGRAM)
try: _s.connect(('8.8.8.8',80)); _ip=_s.getsockname()[0]
except OSError: _ip='localhost'
finally: _s.close()
print(f'3D viewer: http://{_ip}:8000/' if viewer_up() else 'viewer down')

3D viewer: http://10.0.0.20:8000/


# Kill

In [1]:
# ── KILL the notebook viewer server (frees port 8000) ──
!sudo fuser -k 8000/tcp 2>/dev/null || echo 'port 8000 already free'

port 8000 already free


## Scene initialization

In [2]:
import sys, importlib, pkgutil
from pathlib import Path
import time

from workspace.workspace import Workspace
from workspace.bt.launcher import load_recipes

# Walk up to the project root (the folder holding launch.yaml).
PROJ = Path.cwd()
while not (PROJ / 'launch.yaml').exists():
    PROJ = PROJ.parent

# Project-local components (anode, cathode, ...) must register before
# the scene loads — same import main.py does.
sys.path.insert(0, str(PROJ))
for mod in pkgutil.iter_modules([str(PROJ / 'components')]):
    if not mod.name.startswith('_'):
        importlib.import_module(f'components.{mod.name}')

# The production scene pair — same files launch.yaml lists.
scene = [
    str(PROJ / 'scene' / 'core_500.j2'),
    str(PROJ / 'scene' / 'layout.j2'),
]

# port 8000 — the notebook's OWN viewer server; 5000 stays the orchestrator's.
workspace = Workspace(config_path=scene, port=8000)
core = workspace.components['core']
rt = workspace.rt
rcp = load_recipes(workspace, core, PROJ / 'recipes.j2')


❌ core connect @ 10.0.1.10 failed
🔵 core simulation api enabled
[Display] socket.io connected
[Display] 3D viewer: http://10.0.0.20:8000/
[Display] sending initial snapshot (42 items)
[Display] Running at 60 fps


## Simulation on or off

`True` -> SimulationAPI, no hardware. `False` -> the real robot.

In [ ]:
core.simulation(False)

## Robot up — motors on, rail homed

In [3]:
rt.op(state='Starting')
rt.motor(1)
if core.has_rail:
    rt.step('homing rail')
    if not rcp['robot'].set_axis_with_stop(core.rail_cfg):
        rt.step('homing failed')
        raise RuntimeError('rail homing failed')


[STEP][info] homing rail


## Pick a disc

Discs are runtime stock in this project — spawn one into `in_1[A1]`
and lift it with the actions.py suction offsets.

In [4]:
SLOT       = 'A1'
PICK_TCP_Z = -5      # suction drives deeper to grab (actions.py)

if 'disc_dev' not in workspace.components:
    holder = [n for n in workspace.components if 'holder' in n and 'in_1' in n][0]
    workspace.add_component('disc_dev', {
        'type': 'disc_22mm',
        'attach': {'parent_name': holder, 'parent_solid': 'body',
                   'parent_anchor': SLOT, 'child_solid': 'body',
                   'child_anchor': 'center', 'offset': [0, 0, 0, 0, 0, 0]}})
rt.step(f'pick disc from in_1[{SLOT}]')
rcp['disc_in_1'].pick(SLOT, tool_tcp_z_offset=PICK_TCP_Z, soft_approach=True)
print('disc in the gripper')


[STEP][info] pick disc from in_1[A1]
[fold] ik0 54 ms, plan 67 ms, sample 20 ms, blend 66 ms (28 pts)
[traj] smove certified: vaj [240, 281, 3000] (req [240, 800]), motion 1.7s, in 265 ms
[fold] certify 271 ms, send 1701 ms
[fusion] hold: Rack (1 leg(s))
disc in the gripper


## To the camera

One planned travel to the present pose. No read, no exit — the robot
holds there until you run the return cell.

In [6]:
rt.step('present to the bottom camera')
rcp['inspector'].present(approach=True, soft_approach=False)
print('at the present pose')


[STEP][info] present to the bottom camera
[fold] ik0 47 ms, plan 39 ms, sample 32 ms, blend 62 ms (22 pts)
[traj] smove certified: vaj [57, 4, 3000] (req [240, 800]), motion 4.9s, in 122 ms
[fold] certify 132 ms, send 4934 ms
at the present pose


## Return the disc

In [ ]:
rt.step(f'disc back to in_1[{SLOT}]')
rcp['disc_in_1'].place(SLOT, gravity_offset=-5, soft_approach=True)
if 'disc_dev' in workspace.components:
    workspace.remove_component('disc_dev')
core.tail_flush(reason='end of pass')
print('done')
